# Lab 3: Choosing a model with a decision matrix
**Module 2.1: Choosing models.**  **Time:** 30 minutes.

**Goal:** make model choice a documented, repeatable decision instead of a preference.
You will (1) measure two models on the same small test, then (2) score options against weighted criteria.

In [ ]:
# SETUP: run this cell first. It works in Google Colab and in VS Code.
import os, sys, subprocess
REPO_URL = "https://github.com/cybertide-solutions/ai-platform-architect-labs.git"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/course"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/course"], check=True)
    os.chdir("/content/course")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                      # VS Code starts in notebooks/; the course files are one level up
sys.path.insert(0, os.getcwd())
import labkit as lk
lk.setup();

## 1. Measure, don't guess
Four typical enterprise tasks. Each runs on the big and the small model.

In [ ]:
tasks = {
    "summarise": "Summarise in one sentence: Our Q3 cloud bill rose 38% because three teams moved analytics "
                 "workloads to on-demand instances without reserved capacity.",
    "extract": "Return JSON with vendor, amount_inr, due_date from: 'Invoice from Kaveri Logistics for INR 84,500, due 15 Nov 2026.'",
    "reason": "A refund of INR 7,500 is requested. Agents can approve up to INR 5,000, team leads above that. "
              "Who must approve? Answer in one sentence.",
    "classify": "Classify the sentiment (positive, neutral, negative) of: 'Delivery was late again, third time this month.'",
}
rows = []
for name, prompt in tasks.items():
    for m in ["big", "small"]:
        out = lk.chat(prompt, model=m, max_tokens=150, tag=f"m3:{name}:{m}")
        last = lk.CALLS[-1]
        rows.append({"task": name, "model": lk.model_name(m), "latency_s": last["latency_s"],
                     "tokens": last["prompt_tokens"] + last["completion_tokens"],
                     "est_cost_usd": last["est_cost_usd"], "output": out[:90]})
import pandas as pd
results = pd.DataFrame(rows)
results

In [ ]:
results.groupby("model")[["latency_s", "tokens", "est_cost_usd"]].mean().round(4)

**Read the outputs.** Where is the small model good enough? Where does it fail?
That judgement is the "quality" score in the matrix below.

## 2. The decision matrix
Edit the weights for **your** use case (they must add up to 1.0) and the scores (1 = poor, 5 = excellent).

In [ ]:
weights = {
    "quality": 0.30,
    "latency": 0.15,
    "cost": 0.20,
    "data_residency": 0.20,   # can data stay in India / in our cloud tenancy?
    "lock_in_risk": 0.15,     # 5 = easy to switch away
}
options = {
    "Hosted large open model (API)":      {"quality": 4, "latency": 4, "cost": 4, "data_residency": 2, "lock_in_risk": 4},
    "Hosted small open model (API)":      {"quality": 3, "latency": 5, "cost": 5, "data_residency": 2, "lock_in_risk": 4},
    "Cloud provider model (Vertex/Bedrock/Azure)": {"quality": 5, "latency": 4, "cost": 3, "data_residency": 4, "lock_in_risk": 2},
    "Self-hosted open model (own GPUs)":  {"quality": 3, "latency": 3, "cost": 2, "data_residency": 5, "lock_in_risk": 5},
}
assert abs(sum(weights.values()) - 1.0) < 1e-6, "weights must add up to 1.0"
matrix = pd.DataFrame(options).T
matrix["weighted_score"] = sum(matrix[c] * w for c, w in weights.items())
matrix.sort_values("weighted_score", ascending=False)

In [ ]:
ax = matrix["weighted_score"].sort_values().plot.barh(title="Weighted score (higher is better)", figsize=(7, 3))

## Try this
1. Your use case is **customer support chat for a bank** where data must stay in India. Change the weights.
   Does the winner change?
2. Your use case is **internal document summarisation** with 50,000 calls a day. Change the weights again.

**Take away:** the matrix does not decide for you. It makes the trade-off visible and recordable,
which is exactly what an Architecture Decision Record (Module 8) needs.